In [15]:
import pandas as pd
import os
from tqdm import tqdm
from catboost import CatBoostClassifier, Pool
from datetime import date, timedelta

In [ ]:
test_start_date = date(2024, 8, 1)
val_start_date = date(2024, 7, 1)
val_end_date = date(2024, 7, 31)
train_end_date = date(2024, 6, 30)

In [2]:
def add_info(df: pd.DataFrame) -> pd.DataFrame:
    action_info = pd.read_csv('./predict-user-fresh-order/action_type_info.csv')
    widget_info = pd.read_csv('./predict-user-fresh-order/widget_info.csv')
    
    return df.merge(action_info, how='left', on='action_type_id').merge(widget_info, how='left', on='widget_name_id').drop(columns=['action_type_id', 'widget_name_id'])

In [3]:
df_actions = pd.DataFrame()

for file in tqdm(os.listdir('./predict-user-fresh-order/actions_history')):
    if file == "_SUCCESS":
        continue
    df_tmp = pd.read_parquet(f'./predict-user-fresh-order/actions_history/{file}')
    df_actions = pd.concat([df_actions, df_tmp])
    
df_actions = add_info(df_actions)

100%|██████████| 54/54 [00:17<00:00,  3.12it/s]


In [4]:
df_actions.head()

,user_id,timestamp,product_id,page_product_id,action_type,widget_name
0,11147114,2024-06-16 16:58:02,33871158,NaN,to_cart,search_catalog_listing
1,2426498,2024-07-23 18:45:28,584336681,NaN,to_cart,search_catalog_listing
2,836386,2024-03-08 12:13:12,228535577,NaN,to_cart,search_catalog_listing
3,1829415,2024-05-13 17:25:09,428674301,NaN,to_cart,search_catalog_listing
4,6150669,2024-03-21 17:54:19,585572009,NaN,to_cart,search_catalog_listing


In [5]:
df_actions.isna().mean()

user_id            0.00000
timestamp          0.00000
product_id         0.00000
page_product_id    0.87828
action_type        0.00000
widget_name        0.00000
dtype: float64

In [14]:
df_actions['action_type'].value_counts()

action_type
to_cart     79660285
click       66968540
order       31306914
favorite     4065805
Name: count, dtype: int64

In [6]:
df_search_history = pd.DataFrame()

for file in tqdm(os.listdir('./predict-user-fresh-order/search_history')):
    if file == "_SUCCESS":
        continue
    df_tmp = pd.read_parquet(f'./predict-user-fresh-order/search_history/{file}')
    df_search_history = pd.concat([df_search_history, df_tmp])
    
df_search_history = add_info(df_search_history)

100%|██████████| 33/33 [00:17<00:00,  1.84it/s]


In [7]:
df_search_history.head()

,user_id,timestamp,search_query,action_type,widget_name
0,2196785,2024-06-20 20:54:36,вода,search,search_suggestions
1,10577554,2024-03-16 10:16:18,молоко,search,search_suggestions
2,5649193,2024-07-12 18:28:46,белый красивый топ,search,search_suggestions
3,3898950,2024-04-10 14:42:23,масло сливочное,search,search_suggestions
4,6385723,2024-03-18 04:29:58,нескафе дольче густо капсулы,search,search_suggestions


In [8]:
df_search_history.describe()

,user_id,timestamp
count,7.816084e+07,78160845
mean,5.475537e+06,2024-05-14 11:41:02.565860352
min,7.000000e+00,2019-01-02 14:43:56
25%,2.483542e+06,2024-04-05 11:39:37
50%,5.381041e+06,2024-05-13 15:47:20
75%,8.281723e+06,2024-06-21 10:19:09
max,1.118418e+07,2024-07-31 23:59:59
std,3.225672e+06,NaN


In [9]:
df_search_history.isna().mean()

user_id         0.000000
timestamp       0.000000
search_query    0.002593
action_type     0.000000
widget_name     0.000000
dtype: float64

In [10]:
df_products = pd.read_csv('./predict-user-fresh-order/product_information.csv')

In [11]:
df_products.head()

,product_id,name,brand,type,category_id,category_name,price,discount_price
0,26176363,Развивающие тесты (3-4 года) (нов.обл.) | Земц...,Machaon,Печатная книга: Развитие детей,780,Книги,380.0,274.0
1,29898500,Mexx Туалетная вода Ice Touch Man 50 мл,Mexx,Туалетная вода,117,Мужская,2645.0,1859.0
2,33967827,64 ГБ USB Флеш-накопитель USB 3.0/3.1 Gen1 Sma...,SmartBuy,USB-флеш-накопитель,178,Флешки и CD-R,1690.0,469.0
3,135938830,"Чай листовой чёрный Ahmad Tea Classic, 200 г",Ahmad Tea,Чай листовой,465,Чай листовой,319.0,244.0
4,137920686,Seagate 4 ТБ Внешний жесткий диск (STEA4000400...,Seagate,Внешний жесткий диск,615,Внешние жесткие диски,28590.0,9539.0


In [12]:
df_products.describe()

,product_id,category_id,price,discount_price
count,2.384430e+05,238443.000000,238443.000000,238443.000000
mean,8.631651e+08,492.596843,4076.515923,2466.872716
std,5.809518e+08,253.129495,12397.160514,9342.257106
min,1.186260e+05,1.000000,0.000000,0.000000
25%,2.786539e+08,319.000000,500.000000,287.000000
50%,7.154962e+08,539.000000,1214.000000,607.000000
75%,1.532873e+09,691.000000,3149.000000,1485.000000
max,1.677292e+09,938.000000,950000.000000,850000.000000


In [13]:
df_products.isna().mean()

product_id        0.000000
name              0.000063
brand             0.008199
type              0.000000
category_id       0.000000
category_name     0.000604
price             0.000000
discount_price    0.000000
dtype: float64